# Hafta 10 - Yapay Sinir Ağları ile Regresyon

Bu defterde California Housing veri seti üzerinde regresyon problemi çözeceğiz.

**Karşılaştırma:**
- Klasik yöntem: Scikit-learn LinearRegression
- Derin öğrenme: Keras ANN

**Metrikler:** RMSE (Kök Ortalama Kare Hata) ve R² (Belirlilik Katsayısı)

## 1. Kütüphaneler

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `matplotlib` | Grafik ve görselleştirme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |
| `pandas` | Veri çerçeveleri (DataFrame) ile veri analizi |
| `sklearn` | Makine öğrenmesi algoritmaları ve araçları |
| `tensorflow` | Derin öğrenme modelleri oluşturma ve eğitme |
| `warnings` | Uyarı mesajlarını yönetme |


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow sürümü: {tf.__version__}")

## 2. California Housing Veri Seti

Bu veri seti California'daki ev fiyatlarını içerir.

**Özellikler:**
- MedInc: Medyan gelir
- HouseAge: Evin yaşı
- AveRooms: Ortalama oda sayısı
- AveBedrms: Ortalama yatak odası sayısı
- Population: Nüfus
- AveOccup: Ortalama hane halkı büyüklüğü
- Latitude: Enlem
- Longitude: Boylam

**Hedef:** Medyan ev fiyatı (100.000$ biriminde)

In [ ]:
# Veri setini yükle
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

print(f"Veri seti boyutu: {X.shape}")
print(f"Hedef değişken aralığı: [{y.min():.2f}, {y.max():.2f}]")
print(f"\nÖzellikler:")
X.describe().round(2)

### Dağılım Görselleştirmesi

Aşağıdaki grafikte verinin dağılımını histogram ile inceliyoruz. Dağılımın şekli (normal, çarpık, bimodal) hangi istatistiksel yöntemlerin uygulanabileceğini belirler.

In [ ]:
# Veri dağılımını görselleştir
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Özellik Dağılımları', fontsize=14, fontweight='bold')

for i, col in enumerate(X.columns):
    ax = axes[i // 4, i % 4]
    ax.hist(X[col], bins=40, color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_title(col, fontsize=11)
    ax.set_ylabel('Frekans')

plt.tight_layout()
plt.show()

# Hedef değişken dağılımı
plt.figure(figsize=(8, 4))
plt.hist(y, bins=50, color='coral', edgecolor='black', alpha=0.7)
plt.title('Hedef Değişken Dağılımı (Ev Fiyatı)', fontsize=13, fontweight='bold')
plt.xlabel('Fiyat (100.000$)')
plt.ylabel('Frekans')
plt.grid(True, alpha=0.3)
plt.show()

## 3. Veri Ön İşleme

### Eğitim ve Test Setlerine Ayırma

Veriyi eğitim ve test olarak ikiye bölüyoruz. `stratify` parametresi, her iki sette de sınıf dağılımının aynı kalmasını sağlar. `random_state` ile tekrarlanabilir sonuçlar elde ediyoruz.

In [ ]:
# Eğitim/test bölme
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Özellik ölçeklendirme
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Eğitim seti: {X_train_scaled.shape}")
print(f"Test seti: {X_test_scaled.shape}")

## 4. Yöntem 1: Doğrusal Regresyon (sklearn)

### Model Eğitimi

Aşağıdaki kodda modeli eğitim verisi üzerinde eğitiyoruz (`.fit()`). Eğitim sonrası test verisi üzerinde tahmin yapıp (`.predict()`) başarı metriklerini hesaplıyoruz.

In [ ]:
# Doğrusal Regresyon modeli
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Tahminler
y_pred_lr = lr_model.predict(X_test_scaled)

# Metrikler
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("=== Doğrusal Regresyon Sonuçları ===")
print(f"RMSE: {rmse_lr:.4f}")
print(f"R² Skoru: {r2_lr:.4f}")

## 5. Yöntem 2: Yapay Sinir Ağı (Keras)

ANN mimarisi:
- Dense(64, relu) → Dense(32, relu) → Dense(1)
- EarlyStopping ile erken durdurma

In [ ]:
# ANN modeli
ann_model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)  # Regresyon: aktivasyonsuz çıktı
])

ann_model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

ann_model.summary()

### Model Eğitimi

Aşağıdaki kodda modeli eğitim verisi üzerinde eğitiyoruz (`.fit()`). Eğitim sonrası test verisi üzerinde tahmin yapıp (`.predict()`) başarı metriklerini hesaplıyoruz.

In [ ]:
# EarlyStopping callback
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Modeli eğit
history = ann_model.fit(
    X_train_scaled, y_train,
    epochs=100,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

print(f"\nEğitim {len(history.history['loss'])} epoch sürdü.")

### ANN tahminleri

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# ANN tahminleri
y_pred_ann = ann_model.predict(X_test_scaled, verbose=0).flatten()

# Metrikler
rmse_ann = np.sqrt(mean_squared_error(y_test, y_pred_ann))
r2_ann = r2_score(y_test, y_pred_ann)

print("=== Yapay Sinir Ağı Sonuçları ===")
print(f"RMSE: {rmse_ann:.4f}")
print(f"R² Skoru: {r2_ann:.4f}")

## 6. Eğitim Eğrileri

### Dağılım Görselleştirmesi

Aşağıdaki grafikte verinin dağılımını histogram ile inceliyoruz. Dağılımın şekli (normal, çarpık, bimodal) hangi istatistiksel yöntemlerin uygulanabileceğini belirler.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# MSE kaybı
ax1.plot(history.history['loss'], label='Eğitim Kaybı (MSE)', linewidth=2)
ax1.plot(history.history['val_loss'], label='Doğrulama Kaybı (MSE)', linewidth=2)
ax1.set_title('Model Kaybı (MSE)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE')
ax1.legend()
ax1.grid(True, alpha=0.3)

# MAE
ax2.plot(history.history['mae'], label='Eğitim MAE', linewidth=2)
ax2.plot(history.history['val_mae'], label='Doğrulama MAE', linewidth=2)
ax2.set_title('Model MAE', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MAE')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Karşılaştırma: Doğrusal Regresyon vs ANN

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Karşılaştırma tablosu
print("\n" + "="*50)
print(f"{'Metrik':<20} {'Doğrusal Regresyon':>15} {'ANN':>10}")
print("="*50)
print(f"{'RMSE':<20} {rmse_lr:>15.4f} {rmse_ann:>10.4f}")
print(f"{'R² Skoru':<20} {r2_lr:>15.4f} {r2_ann:>10.4f}")
print("="*50)

kazanan = "ANN" if rmse_ann < rmse_lr else "Doğrusal Regresyon"
print(f"\nDaha düşük RMSE'ye sahip model: {kazanan}")

# Karşılaştırma grafiği
fig, ax = plt.subplots(figsize=(8, 5))
modeller = ['Doğrusal\nRegresyon', 'Yapay Sinir\nAğı (ANN)']
rmse_degerleri = [rmse_lr, rmse_ann]
r2_degerleri = [r2_lr, r2_ann]

x = np.arange(len(modeller))
width = 0.35

bars1 = ax.bar(x - width/2, rmse_degerleri, width, label='RMSE', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, r2_degerleri, width, label='R²', color='#2ecc71', alpha=0.8)

ax.set_title('Model Karşılaştırması', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(modeller, fontsize=11)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 8. Gerçek vs Tahmin Görselleştirmesi

### Saçılım Grafiği

İki değişken arasındaki ilişkiyi saçılım grafiği ile inceliyoruz. Noktaların oluşturduğu desen, doğrusal veya doğrusal olmayan ilişkiyi gösterir.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Doğrusal Regresyon
ax1.scatter(y_test, y_pred_lr, alpha=0.3, s=10, color='steelblue')
ax1.plot([0, 5.5], [0, 5.5], 'r--', linewidth=2, label='Mükemmel Tahmin Çizgisi')
ax1.set_title(f'Doğrusal Regresyon\nRMSE={rmse_lr:.3f}, R²={r2_lr:.3f}',
              fontsize=12, fontweight='bold')
ax1.set_xlabel('Gerçek Fiyat', fontsize=11)
ax1.set_ylabel('Tahmin Edilen Fiyat', fontsize=11)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 5.5)
ax1.set_ylim(0, 5.5)

# ANN
ax2.scatter(y_test, y_pred_ann, alpha=0.3, s=10, color='coral')
ax2.plot([0, 5.5], [0, 5.5], 'r--', linewidth=2, label='Mükemmel Tahmin Çizgisi')
ax2.set_title(f'Yapay Sinir Ağı (ANN)\nRMSE={rmse_ann:.3f}, R²={r2_ann:.3f}',
              fontsize=12, fontweight='bold')
ax2.set_xlabel('Gerçek Fiyat', fontsize=11)
ax2.set_ylabel('Tahmin Edilen Fiyat', fontsize=11)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 5.5)
ax2.set_ylim(0, 5.5)

plt.tight_layout()
plt.show()

### Dağılım Görselleştirmesi

Aşağıdaki grafikte verinin dağılımını histogram ile inceliyoruz. Dağılımın şekli (normal, çarpık, bimodal) hangi istatistiksel yöntemlerin uygulanabileceğini belirler.

In [ ]:
# Hata dağılımı karşılaştırması
errors_lr = y_test - y_pred_lr
errors_ann = y_test - y_pred_ann

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(errors_lr, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax1.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax1.set_title(f'Doğrusal Regresyon Hata Dağılımı\nOrtalama: {errors_lr.mean():.4f}',
              fontsize=12, fontweight='bold')
ax1.set_xlabel('Hata (Gerçek - Tahmin)')
ax1.set_ylabel('Frekans')
ax1.grid(True, alpha=0.3)

ax2.hist(errors_ann, bins=50, color='coral', edgecolor='black', alpha=0.7)
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax2.set_title(f'ANN Hata Dağılımı\nOrtalama: {errors_ann.mean():.4f}',
              fontsize=12, fontweight='bold')
ax2.set_xlabel('Hata (Gerçek - Tahmin)')
ax2.set_ylabel('Frekans')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Özet

Bu defterde:
- California Housing veri setini keşfettik
- Doğrusal Regresyon ile temel tahmin yaptık
- ANN ile regresyon modeli oluşturduk (EarlyStopping ile)
- İki yaklaşımı RMSE ve R² metrikleriyle karşılaştırdık
- Gerçek vs tahmin grafiklerini çizdik

**Önemli çıkarımlar:**
- ANN, doğrusal olmayan ilişkileri yakalayabilir
- EarlyStopping, aşırı öğrenmeyi önler
- Özellik ölçeklendirme (StandardScaler) ANN için kritiktir